# testing and troubleshooting bug with nested json.

Hypotesen är att `payload` i bronze är en nästlad dict struktur men att PyArrow konverterar den till någon annan vid lösning. Antingen en sträng eller ett PyArrow specifikt objekt.

In [29]:
import pandas as pd

# Läs hela Silver-mappen, pandas hittar alla parquet-filer automatiskt
silver_df = pd.read_parquet("data/silver/events/")

print(f"Antal rader i Silver: {len(silver_df)}")
print(f"Kolumner: {list(silver_df.columns)}")

Antal rader i Silver: 15803
Kolumner: ['event_id', 'event_type', 'actor_login', 'repo_name', 'repo_id', 'commit_count', 'pr_action', 'pr_merged', 'created_at', 'year', 'month', 'day']


In [30]:
push_df = silver_df[silver_df["event_type"] == "PushEvent"]

print(f"Antal PushEvents: {len(push_df)}")
print(push_df["commit_count"].describe())
print(push_df["commit_count"].value_counts().head(10))

Antal PushEvents: 11743
count    11743.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: commit_count, dtype: float64
commit_count
0    11743
Name: count, dtype: int64


In [31]:
bronze_df = pd.read_parquet("data/bronze/events/year=2026/month=04/")  # senaste bootstrap-datan

# Hitta en PushEvent och skriv ut payload-strängen som den faktiskt ser ut i Bronze
push_sample = bronze_df[bronze_df["type"] == "PushEvent"].iloc[0]

import json
payload_str = push_sample["payload"]
payload_dict = json.loads(payload_str)  # Deserialisera strängen

print("Alla nycklar i payload:")
print(list(payload_dict.keys()))

print("\nFinns 'size' i payload?", "size" in payload_dict)
print("Finns 'commits' i payload?", "commits" in payload_dict)

print("\nHela payload:")
print(json.dumps(payload_dict, indent=2))

Alla nycklar i payload:
['repository_id', 'push_id', 'ref', 'head', 'before']

Finns 'size' i payload? False
Finns 'commits' i payload? False

Hela payload:
{
  "repository_id": 1191939486,
  "push_id": 32386342257,
  "ref": "refs/heads/main",
  "head": "bd6e1c04be5993aaa0151663ff99f0932ceb7090",
  "before": "f64aa871f2f3676afda9962ce46b4ea030b53a38"
}


In [32]:
# Läs specifikt en live-fil, inte bootstrap
part_df = pd.read_parquet("data/bronze/events/year=2026/month=04/day=03/")

# Filtrera fram bara part-filer om det finns både bootstrap och part i samma mapp
# Annars läs hela mappen

push_sample = part_df[part_df["type"] == "PushEvent"].iloc[0]

print("Payload-typ i live-fil:", type(push_sample["payload"]))
print("Payload-innehåll:", push_sample["payload"])

Payload-typ i live-fil: <class 'str'>
Payload-innehåll: {"repository_id": 1168935803, "push_id": 32433577299, "ref": "refs/heads/master", "head": "9f9fd2d49c745de182bbf2672e2cfef9364e8568", "before": "ce75730f50f537eb0f4dbb29d246021093f0fcfd"}


## Hypotesen stämmer!
Förklaring:

Nu syns buggen tydligt. Titta på hela payload-innehållet som kom tillbaka ifrån ovanstående cell som körs på en Bootstrap fil ifrån Github Archives. I den finns ingen commit-nyckel. Inte None, inte en tom lista, utan den är helt frånvarande. Det är DET som är själva problemet.
Det som hände är att PyArrow, när den körde pa.Table.from_pylist(events) i consumer och skapade Parquet-schemat, stötte på commits som en lista av objekt med varierande struktur. PyArrow är bra på att inferera schema för enkla typer men kan tappa eller omstrukturera djupt nästade listor på oväntade sätt, särskilt när olika events i samma batch har olika struktur i payload.

Min _flatten()-funktion gör alltså exakt vad den ska, den ber om event.get("payload", {}).get("commits", []), men den hittar aldrig en commits-nyckel eftersom den försvann redan när Bronze-filen skapades. Resultatet är len([]) = 0 varje gång, tyst och utan fel.

Det är den farligaste typen av bug i en pipeline: ingen traceback, ingen varning, bara felaktig data som ser korrekt ut.

---

För att bekräfta allting ÄNNU en gång blir det mer utforskning

## Hypotes 2:

hypotes 2 är att commits-listan antingen försvann vid PyArrows schema-inferens, eller att den lagrades som en separat kolumn på toppnivå i Parquet-filen snarare än nästlad inuti payload. Det är ett klassiskt beteende när PyArrow "plattar ut" schema automatiskt. Det kommer nycklarna här nedan bevisa:

In [14]:
push_df = silver_df[silver_df["event_type"] == "PushEvent"]

print(f"Antal PushEvents: {len(push_df)}")
print(push_df["commit_count"].describe())
print(push_df["commit_count"].value_counts().head(10))

Antal PushEvents: 5647
count    5647.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: commit_count, dtype: float64
commit_count
0    5647
Name: count, dtype: int64


## Bug discovered and root cause of bug is found.

Buggen är inte där jag trodde den var från början, dvs i `bronze_to_silver.py` i `def _flatten()` funktionen utan så djupt ner i ledet det bara går. Buggen finns i `consumer.py`


---

Förklaring:

Buggen sitter i `_write_batch_to_bronze()` i `consumer.py`. 

Fixen: ett enda steg som läggs till innan `pa.Table.from_pylist():`. Och i _flatten() i bronze_to_silver.py behöver jag lägga till json.loads() för att deserialisera strängen tillbaka till en dict.

In [15]:
bronze_df = pd.read_parquet("data/bronze/events/year=2026/month=04/")  # senaste bootstrap-datan

# Hitta en PushEvent och skriv ut payload-strängen som den faktiskt ser ut i Bronze
push_sample = bronze_df[bronze_df["type"] == "PushEvent"].iloc[0]

import json
payload_str = push_sample["payload"]
payload_dict = json.loads(payload_str)  # Deserialisera strängen

print("Alla nycklar i payload:")
print(list(payload_dict.keys()))

print("\nFinns 'size' i payload?", "size" in payload_dict)
print("Finns 'commits' i payload?", "commits" in payload_dict)

print("\nHela payload:")
print(json.dumps(payload_dict, indent=2))

Alla nycklar i payload:
['repository_id', 'push_id', 'ref', 'head', 'before']

Finns 'size' i payload? False
Finns 'commits' i payload? False

Hela payload:
{
  "repository_id": 140395025,
  "push_id": 32427581937,
  "ref": "refs/heads/trunk",
  "head": "ede01b871ec93112e85589b3d70189ab97d06adf",
  "before": "82869a77bb68a6f40e3111e69834423f1a8d6c99"
}


In [27]:

# Läs in en silver fil
silver_df = pd.read_parquet("data/silver/events/year=2026/month=04/day=02/part-20260403_205335_351669.parquet"
)

# Ta en PushEvent och undersök raw formatet
silver_sample = silver_df[silver_df["type"] == "PushEvent"].iloc[0]

# Vad är typen på Payload just NU, i bronze layer?

print(type(silver_sample["payload"]))
print(silver_sample["payload"])

KeyError: 'type'